# Week 7 (starter): The Adaptation Decision and a Production Dataset

No GPU, no fine-tuning. The dataset and the dynamic few-shot selection run locally; the evaluation calls Gemini (guarded: without a key it reports API-BLOCKED, the local parts still run). Cells marked **TODO (you)** are yours. Dependencies: `tiktoken`, `sentence-transformers`. For the eval: `pip install openai`.

In [ ]:
import os, re, random, pathlib
from collections import Counter
random.seed(7)
# TODO (you): replace this generator with your own domain data. Task here: intent classification.
INTENTS={'set_reminder':('remind me to {task} {time}','set a reminder to {task}','ping me to {task} {time}','reminder: {task} {time}'),
  'get_weather':('weather in {city}','will it rain in {city} {time}','how hot in {city}','forecast for {city} {time}'),
  'play_music':('play {x}','put on some {x}','queue up {x}','I want to hear {x}'),
  'send_message':('text {p} that {task}','message {p} about {task}','tell {p} {task}','let {p} know {task}'),
  'get_directions':('directions to {city}','navigate to {city} {time}','route to {city}','take me to {city}')}
SLOT={'task':['call mom','buy milk','submit the report','pay rent','book a flight','walk the dog','renew the license','email the client','restock coffee','back up the laptop'],
  'time':['tomorrow','at 5pm','tonight','this weekend','on monday','at noon','next week','before bed'],
  'city':['Denver','Tokyo','Lagos','Paris','Cairo','Oslo','Lima','Seoul','Madrid','Dublin'],
  'x':['jazz','lofi','Adele','Miles Davis','reggaeton','classical','Fela Kuti','ambient'],
  'p':['Sam','my brother','the team','Priya','Dad','Marcus','my sister','the group']}
def fill(t): return re.sub(r'\{(\w+)\}', lambda m: random.choice(SLOT[m.group(1)]), t)
raw=[{'text':fill(random.choice(tmpl)),'label':lab} for lab,tmpl in INTENTS.items() for _ in range(140)]
raw+=[{'text':'','label':'set_reminder'}]*4+[{'text':'hi','label':'unknown'}]*3   # noise to clean
print('raw pool:', len(raw))

In [ ]:
VALID=set(INTENTS)
valid=[r for r in raw if r['text'].strip() and r['label'] in VALID]
seen=set(); dedup=[]
for r in valid:
    k=(r['text'].lower(),r['label'])
    if k not in seen: seen.add(k); dedup.append(r)
print(f'dropped invalid {len(raw)-len(valid)}, dropped duplicate {len(valid)-len(dedup)}, clean {len(dedup)}')
by={lab:[r for r in dedup if r['label']==lab] for lab in VALID}
train=[]; eval_=[]
for lab,rows in by.items():
    random.shuffle(rows); train+=rows[:40]; eval_+=rows[40:50]
random.shuffle(train); random.shuffle(eval_)
print('train',len(train),'eval',len(eval_))
print('train label balance:', dict(Counter(r['label'] for r in train)))
import tiktoken
os.environ['TIKTOKEN_CACHE_DIR']=str((pathlib.Path('.')/'.tiktoken_cache').resolve())
enc=tiktoken.get_encoding('cl100k_base'); lens=[len(enc.encode(r['text'])) for r in train]
print(f'token length  min {min(lens)} median {sorted(lens)[len(lens)//2]} max {max(lens)}')
print('format check (text+label present):', all(r['text'] and r['label'] for r in train+eval_))

## Part 2: Adapt with dynamic few-shot (local selection)
For each eval input, retrieve the most similar training examples to place in the prompt. This is the no-GPU adaptation, and it reuses Week 5 retrieval.

In [ ]:
os.environ['HF_HOME']=str((pathlib.Path('.')/'.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
train_emb=emb.encode([r['text'] for r in train], convert_to_tensor=True, normalize_embeddings=True)
def few_shot(query, k=4):
    qe=emb.encode([query], convert_to_tensor=True, normalize_embeddings=True)
    sims=util.cos_sim(qe, train_emb).cpu().numpy()[0]
    return [train[i] for i in sims.argsort()[::-1][:k]]
ex=eval_[0]
print('eval input:', ex['text'], '(true:', ex['label'], ')')
print('retrieved few-shot examples:')
for s in few_shot(ex['text'], 4): print('  ', s['text'], '->', s['label'])

In [ ]:
def gemini_label(prompt, model='gemini-2.5-flash-lite'):
    if not os.environ.get('GEMINI_API_KEY'): return None
    from openai import OpenAI
    c=OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return c.chat.completions.create(model=model, messages=[{'role':'user','content':prompt}]).choices[0].message.content.strip()

def classify(text, shots=None):
    head='Classify the intent into one of: '+', '.join(VALID)+'.\n'
    body=''.join(f'Text: {s["text"]}\nIntent: {s["label"]}\n' for s in (shots or []))
    return gemini_label(head+body+f'Text: {text}\nIntent:')

if os.environ.get('GEMINI_API_KEY'):
    def acc(use_shots):
        ok=0
        for r in eval_:
            pred=classify(r['text'], few_shot(r['text']) if use_shots else None)
            ok += (pred or '').strip().lower().startswith(r['label'])
        return ok/len(eval_)
    print(f'zero-shot accuracy: {acc(False):.0%}')
    print(f'dynamic few-shot accuracy: {acc(True):.0%}')
else:
    print('[API-BLOCKED] set GEMINI_API_KEY to run the zero-shot vs dynamic-few-shot evaluation.')
    print('The dataset and the example selection above are real; only the scored eval needs the model.')
# TODO (you): report before/after, and find one slice where few-shot underperforms zero-shot.

## Part 4 and 5: decision memo, submit
**TODO (you):** write a short memo weighing prompt, RAG, dynamic few-shot, fine-tuning, and doing nothing for your task, grounded in your numbers. Commit the dataset, update your agent context file, open a pull request with a research note. Rubric: dataset + quality report (30), dynamic few-shot (25), before/after + failure slice (20), decision memo (15), PR hygiene (10).